# Caso 05: RutaSabor — Cocina Fantasma: ¿Dónde Está el Margen?
## Día 2: Antes de Calcular, Limpiar (y Decidir tú qué es el Margen)

**Objetivos del Día 2:**
1. **Limpieza Forense con Criterio:**
   - Aplicar el inventario detectado el Día 1 sobre los 8 ficheros crudos sin perder información legítima.
   - Deduplicar los webhooks repetidos de Glovo en `pedidos`, `lineas_pedido` e `incidencias`.
   - Resolver los formatos heterogéneos de fechas (ISO, DD/MM/YYYY, meses en inglés: *Jun*, *Sep*).
   - Convertir importes con símbolo de euro (`€`) y comisiones con porcentaje (`%`) a valores numéricos (`float`).
   - Imputar valores atípicos/texto en importes a partir de las líneas del pedido.
   - Estandarizar categorizaciones y grafías en ciudades y canales.
   - Respetar los nulos con significado: `cliente_id` vacío representa pedidos de **invitados**.
2. **Definición de la Métrica Central:**
   - Formalizar la definición y alcance del **Margen de Contribución por Pedido**.
3. **Primera Consulta SQL Analítica (DuckDB):**
   - Confirmar el titular de superficie de Gonzalo Prats: ¿Es La Brasa Burger la número uno en pedidos y en GMV?


In [1]:
import pandas as pd
import numpy as np
import os
import duckdb

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

RAW_DIR = os.path.join('data', 'raw')
CLEAN_DIR = os.path.join('data', 'clean')
os.makedirs(CLEAN_DIR, exist_ok=True)

print(f"Directorio Origen (Raw): {RAW_DIR}")
print(f"Directorio Destino (Clean): {CLEAN_DIR}")


Directorio Origen (Raw): data\raw
Directorio Destino (Clean): data\clean


In [2]:
def limpiar_moneda(serie):
    """Limpia cadenas con símbolo €, puntos de miles y comas decimales a float."""
    return (
        serie.astype(str)
        .str.replace('€', '', regex=False)
        .str.replace('', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )

def limpiar_porcentaje(serie):
    """Convierte texto de porcentaje ('30 %' o '0,25') a decimal float (0.30)."""
    s = serie.astype(str).str.replace(' ', '', regex=False).str.replace('%', '', regex=False).str.replace(',', '.', regex=False)
    num = pd.to_numeric(s, errors='coerce')
    # Si viene en escala 0-100 (ej. 30), dividir por 100; si viene en 0-1 (ej. 0.25), mantener
    return np.where(num > 1.0, num / 100.0, num)


### 1. Limpieza: `canales.csv`
Estandarizamos el porcentaje de comisión a decimal (`0.30`, `0.28`, `0.25`, `0.06`) y unificamos el tipo de canal.


In [3]:
df_canales = pd.read_csv(os.path.join(RAW_DIR, 'canales.csv'), encoding='utf-8')

# Limpieza
df_canales['comision_pct'] = limpiar_porcentaje(df_canales['comision_pct'])
df_canales['tipo'] = df_canales['tipo'].str.strip().str.upper()
df_canales['tipo'] = df_canales['tipo'].replace({'PROPIO': 'PROPIO', 'MARKETPLACE': 'MARKETPLACE'})

# Persistencia
df_canales.to_csv(os.path.join(CLEAN_DIR, 'canales.csv'), index=False)
df_canales.to_parquet(os.path.join(CLEAN_DIR, 'canales.parquet'), index=False)

print("canales limpio:")
df_canales


canales limpio:


,canal_id,nombre,comision_pct,tipo
0,1,glovo,0.30,MARKETPLACE
1,2,uber_eats,0.28,MARKETPLACE
2,3,just_eat,0.25,MARKETPLACE
3,4,app_propia,0.06,PROPIO


### 2. Limpieza: `marcas.csv`
Parseamos la fecha de lanzamiento a formato ISO estándar (`YYYY-MM-DD`) y normalizamos el tipo de cocina.


In [4]:
df_marcas = pd.read_csv(os.path.join(RAW_DIR, 'marcas.csv'), encoding='utf-8')

# Limpieza
df_marcas['fecha_lanzamiento'] = pd.to_datetime(df_marcas['fecha_lanzamiento'], format='mixed').dt.strftime('%Y-%m-%d')
df_marcas['tipo_cocina'] = df_marcas['tipo_cocina'].str.strip().str.lower()

# Persistencia
df_marcas.to_csv(os.path.join(CLEAN_DIR, 'marcas.csv'), index=False)
df_marcas.to_parquet(os.path.join(CLEAN_DIR, 'marcas.parquet'), index=False)

print("marcas limpio:")
df_marcas


marcas limpio:


,marca_id,nombre,tipo_cocina,fecha_lanzamiento
0,1,La Brasa Burger,hamburguesa,2019-06-01
1,2,Poke Break,poke,2021-03-15
2,3,Pizza Nostra,pizza,2020-10-09
3,4,Wok Express,wok,2022-01-20
4,5,Dulce Antojo,postre,2023-12-05


### 3. Limpieza: `cocinas.csv`
Estandarizamos nombres de ciudades (resolviendo códigos como `MAD`, `BCN`, `VLC`, `ZGZ`), parseamos fechas de apertura con meses en inglés (`1 Jun 2019`, `1 Sep 2023`) y corregimos capacidades centinela.


In [5]:
df_cocinas = pd.read_csv(os.path.join(RAW_DIR, 'cocinas.csv'), encoding='utf-8')

# Mapeo canónico de ciudades
map_ciudades = {
    'MAD': 'Madrid', 'madrid': 'Madrid', 'MADRID': 'Madrid', 'Madrid': 'Madrid',
    'BCN': 'Barcelona', 'barcelona': 'Barcelona', 'BARCELONA': 'Barcelona', 'Barcelona': 'Barcelona',
    'VLC': 'Valencia', 'valencia': 'Valencia', 'VALENCIA': 'Valencia', 'Valencia': 'Valencia',
    'ZGZ': 'Zaragoza', 'zaragoza': 'Zaragoza', 'ZARAGOZA': 'Zaragoza', 'Zaragoza': 'Zaragoza'
}
df_cocinas['ciudad'] = df_cocinas['ciudad'].map(map_ciudades)
df_cocinas['fecha_apertura'] = pd.to_datetime(df_cocinas['fecha_apertura'], format='mixed').dt.strftime('%Y-%m-%d')

# Tratamiento de capacidad centinela (99999 o 0 corregidos por mediana de cocinas de su ciudad)
cap_median = df_cocinas[~df_cocinas['capacidad'].isin([0, 99999])]['capacidad'].median()
df_cocinas['capacidad'] = df_cocinas['capacidad'].replace({0: int(cap_median), 99999: int(cap_median)})

# Persistencia
df_cocinas.to_csv(os.path.join(CLEAN_DIR, 'cocinas.csv'), index=False)
df_cocinas.to_parquet(os.path.join(CLEAN_DIR, 'cocinas.parquet'), index=False)

print("cocinas limpio:")
df_cocinas


cocinas limpio:

,cocina_id,ciudad,nombre,fecha_apertura,capacidad
0,1,Madrid,Chamberi,2019-06-01,520
1,2,Madrid,Vallecas,2023-09-01,480
2,3,Madrid,Tetuan,2020-11-01,500
3,4,Madrid,Arganzuela,2021-04-01,475
4,5,Barcelona,Poblenou,2020-03-01,510
5,6,Barcelona,Sants,2021-07-01,470
6,7,Barcelona,Gracia,2022-02-01,440
7,8,Valencia,Ruzafa,2020-10-01,450
8,9,Valencia,Campanar,2022-06-01,475
9,10,Zaragoza,Delicias,2021-02-01,430


### 4. Limpieza: `productos.csv`
Convertimos precio de venta y coste de ingredientes a float y capitalizamos las categorías.


In [6]:
df_productos = pd.read_csv(os.path.join(RAW_DIR, 'productos.csv'), encoding='utf-8')

df_productos['precio'] = pd.to_numeric(limpiar_moneda(df_productos['precio']), errors='coerce')
df_productos['coste_ingredientes'] = pd.to_numeric(limpiar_moneda(df_productos['coste_ingredientes']), errors='coerce')
df_productos['categoria'] = df_productos['categoria'].str.strip().str.capitalize()

# Persistencia
df_productos.to_csv(os.path.join(CLEAN_DIR, 'productos.csv'), index=False)
df_productos.to_parquet(os.path.join(CLEAN_DIR, 'productos.parquet'), index=False)

print(f"productos limpio: {len(df_productos)} registros")
df_productos.head(3)


productos limpio: 124 registros


,producto_id,marca_id,nombre,precio,coste_ingredientes,categoria
0,1,1,Burger BBQ,3.46,1.95,Principal
1,2,1,Burger Clasica,4.85,2.69,Principal
2,3,1,Burger Doble,5.89,3.23,Principal


### 5. Limpieza: `clientes.csv`
Estandarizamos ciudades, unificamos los canales de registro y normalizamos fechas a ISO.


In [7]:
df_clientes = pd.read_csv(os.path.join(RAW_DIR, 'clientes.csv'), encoding='utf-8')

df_clientes['ciudad'] = df_clientes['ciudad'].map(map_ciudades).fillna(df_clientes['ciudad'].str.capitalize())

map_canales_reg = {
    'app_propia': 'app_propia', 'app': 'app_propia', 'App': 'app_propia',
    'glovo': 'glovo', 'Glovo': 'glovo',
    'web': 'web', 'Web': 'web',
    'uber': 'uber_eats', 'Uber Eats': 'uber_eats'
}
df_clientes['canal_registro'] = df_clientes['canal_registro'].map(map_canales_reg).fillna('desconocido')
df_clientes['fecha_registro'] = pd.to_datetime(df_clientes['fecha_registro'], format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')

# Persistencia
df_clientes.to_csv(os.path.join(CLEAN_DIR, 'clientes.csv'), index=False)
df_clientes.to_parquet(os.path.join(CLEAN_DIR, 'clientes.parquet'), index=False)

print(f"clientes limpio: {len(df_clientes):,} registros")
df_clientes.head(3)


clientes limpio: 60,000 registros


,cliente_id,ciudad,fecha_registro,canal_registro
0,1,Zaragoza,2023-09-26,web
1,2,Valencia,2023-05-19,web
2,3,Valencia,2022-10-14,web


### 6. Limpieza: `incidencias.csv`
Deduplicamos por `incidencia_id`, convertimos la compensación a float y agrupamos los tipos de incidencia en 4 categorías maestras.


In [8]:
df_incidencias = pd.read_csv(os.path.join(RAW_DIR, 'incidencias.csv'), encoding='utf-8')
filas_antes = len(df_incidencias)

# Deduplicar
df_incidencias = df_incidencias.drop_duplicates(subset=['incidencia_id']).copy()
print(f"incidencias: {filas_antes:,} -> {len(df_incidencias):,} ({filas_antes - len(df_incidencias):,} duplicados eliminados)")

# Tipos y categorías
df_incidencias['importe_compensacion'] = pd.to_numeric(limpiar_moneda(df_incidencias['importe_compensacion']), errors='coerce')
df_incidencias['fecha'] = pd.to_datetime(df_incidencias['fecha'], format='mixed', errors='coerce').dt.strftime('%Y-%m-%d')

map_incidencias = {
    'retraso': 'retraso', 'tarde': 'retraso', 'pedido_tarde': 'retraso',
    'frio': 'frio', 'comida fria': 'frio', 'producto_frio': 'frio',
    'erroneo': 'erroneo', 'producto_erroneo': 'erroneo', 'wrong_item': 'erroneo',
    'no entregado': 'no_entregado', 'no_entregado': 'no_entregado', 'missing': 'no_entregado'
}
df_incidencias['tipo_incidencia'] = df_incidencias['tipo_incidencia'].map(map_incidencias).fillna('otro')

# Persistencia
df_incidencias.to_csv(os.path.join(CLEAN_DIR, 'incidencias.csv'), index=False)
df_incidencias.to_parquet(os.path.join(CLEAN_DIR, 'incidencias.parquet'), index=False)

print("incidencias limpio:")
df_incidencias.head(3)


incidencias: 10,077 -> 9,832 (245 duplicados eliminados)


incidencias limpio:


,incidencia_id,pedido_id,tipo_incidencia,importe_compensacion,fecha
0,901000,114569,frio,5.37,2024-05-28
1,902710,140595,erroneo,4.23,2023-09-18
2,900147,102159,retraso,10.01,2024-04-05


### 7. Limpieza: `lineas_pedido.csv`
Deduplicamos por `linea_id` y convertimos `precio_unitario` a float.


In [9]:
df_lineas = pd.read_csv(os.path.join(RAW_DIR, 'lineas_pedido.csv'), encoding='utf-8')
filas_antes = len(df_lineas)

df_lineas = df_lineas.drop_duplicates(subset=['linea_id']).copy()
print(f"lineas_pedido: {filas_antes:,} -> {len(df_lineas):,} ({filas_antes - len(df_lineas):,} duplicados eliminados)")

df_lineas['precio_unitario'] = pd.to_numeric(limpiar_moneda(df_lineas['precio_unitario']), errors='coerce')

# Persistencia
df_lineas.to_csv(os.path.join(CLEAN_DIR, 'lineas_pedido.csv'), index=False)
df_lineas.to_parquet(os.path.join(CLEAN_DIR, 'lineas_pedido.parquet'), index=False)

print("lineas_pedido limpio:")
df_lineas.head(3)


lineas_pedido: 362,032 -> 360,231 (1,801 duplicados eliminados)


lineas_pedido limpio:


,linea_id,pedido_id,producto_id,cantidad,precio_unitario
0,5104895,143711,93,2,3.84
1,5139299,158001,87,1,4.70
2,5269292,212135,20,1,4.95


### 8. Limpieza: `pedidos.csv` (Tabla de Hechos)
1. Deduplicamos por `pedido_id` (de 151.500 a 150.000 pedidos).
2. Limpiamos `importe_cobrado` y `descuento`. Para los 5 registros corruptos de importe (`N/D`, `-`, `error`, `pendiente`), imputamos el total a partir de la suma de `cantidad * precio_unitario` de sus líneas de pedido.
3. Parseamos `fecha_hora` a formato ISO.
4. Normalizamos el campo `estado` (`entregado`, `cancelado`, `en_reparto`).
5. **Preservamos los nulos en `cliente_id`**: representan a 53.308 clientes invitados.


In [10]:
df_pedidos = pd.read_csv(os.path.join(RAW_DIR, 'pedidos.csv'), encoding='utf-8')
filas_antes = len(df_pedidos)

df_pedidos = df_pedidos.drop_duplicates(subset=['pedido_id']).copy()
print(f"pedidos: {filas_antes:,} -> {len(df_pedidos):,} ({filas_antes - len(df_pedidos):,} duplicados de webhook eliminados)")

# Limpieza de importe y descuento
imp_raw = limpiar_moneda(df_pedidos['importe_cobrado'])
df_pedidos['importe_cobrado'] = pd.to_numeric(imp_raw, errors='coerce')
df_pedidos['descuento'] = pd.to_numeric(limpiar_moneda(df_pedidos['descuento']), errors='coerce').fillna(0.0)

# Imputación forense de importes no numéricos a partir de lineas_pedido
nulos_imp = df_pedidos[df_pedidos['importe_cobrado'].isna()]
if not nulos_imp.empty:
    print(f"Imputando {len(nulos_imp)} importes corruptos desde lineas_pedido...")
    df_lineas_tot = df_lineas.groupby('pedido_id').apply(lambda g: (g['cantidad'] * g['precio_unitario']).sum(), include_groups=False).rename('importe_recalculado')
    df_pedidos = df_pedidos.merge(df_lineas_tot, on='pedido_id', how='left')
    df_pedidos['importe_cobrado'] = df_pedidos['importe_cobrado'].fillna(df_pedidos['importe_recalculado'] - df_pedidos['descuento'])
    df_pedidos = df_pedidos.drop(columns=['importe_recalculado'])

# Fechas y estados
df_pedidos['fecha_hora'] = pd.to_datetime(df_pedidos['fecha_hora'], format='mixed', dayfirst=True).dt.strftime('%Y-%m-%d %H:%M:%S')

map_estados = {
    'entregado': 'entregado', 'Entregado': 'entregado', 'ok': 'entregado', 'ENTREGADO': 'entregado',
    'cancelado': 'cancelado', 'Cancelado': 'cancelado', 'cancel': 'cancelado',
    'en_reparto': 'en_reparto', 'en reparto': 'en_reparto', 'En reparto': 'en_reparto'
}
df_pedidos['estado'] = df_pedidos['estado'].map(map_estados).fillna('otro')

# Persistencia
df_pedidos.to_csv(os.path.join(CLEAN_DIR, 'pedidos.csv'), index=False)
df_pedidos.to_parquet(os.path.join(CLEAN_DIR, 'pedidos.parquet'), index=False)

print(f"pedidos limpio: {len(df_pedidos):,} pedidos listos")
df_pedidos.head(3)


pedidos: 151,500 -> 150,000 (1,500 duplicados de webhook eliminados)


Imputando 5 importes corruptos desde lineas_pedido...


pedidos limpio: 150,000 pedidos listos


,pedido_id,ref_externa,cocina_id,marca_id,canal_id,cliente_id,fecha_hora,importe_cobrado,descuento,estado,tiempo_entrega_min
0,169896,PO-346494,1,2,3,59683.0,2023-08-29 20:46:00,17.26,0.00,entregado,NaN
1,205656,WO-144417,5,4,2,NaN,2023-10-20 13:14:00,12.91,0.00,entregado,13.0
2,161632,BU-715696,2,1,1,671.0,2024-01-06 21:47:00,18.41,2.03,entregado,41.0


## Decisión Conceptual: ¿Qué es el Margen de Contribución por Pedido?

En un negocio de restauración tradicional, el cliente paga en el mostrador y los costes se dividen en comida y sala. En una **cocina fantasma con delivery por plataforma**, la economía cambia radicalmente:

$$\\text{Margen de Contribución} = \\text{Importe Cobrado} - \\text{Coste Ingredientes} - \\text{Comisión Marketplace} - \\text{Coste Incidencias}$$

### ¿Por qué esta definición y no Margen Bruto o EBITDA?
1. **Costes directos variables por pedido:**
   * **Ingredientes:** Lo que cuesta la materia prima de lo consumido (receta).
   * **Comisión:** Lo que cobra Glovo/Uber/Just Eat por intermediar y repartir (hasta el 30%).
   * **Incidencias:** Lo que se le devuelve o compensa al cliente cuando el pedido llega frío, tarde o equivocado.
2. **Costes de estructura (Fijos):**
   * Alquiler de las naves, salarios base de la brigada de cocineros, suministros eléctricos de las planchas y software general.
   * **El papel del Margen de Contribución:** Es el dinero neto real que *cada pedido deja en la caja* para pagar la estructura. Si un pedido factura mucho pero deja solo 0,60 € de contribución, **no paga el alquiler: solo mueve cajas**.


## Primera Consulta SQL: Confirmar el Titular de Gonzalo Prats

Gonzalo afirma: *"La Brasa Burger es la número UNO, más pedidos y más facturación que ninguna, de calle"*.
Conectamos DuckDB a las tablas limpias para verificar su afirmación en superficie:


In [11]:
con = duckdb.connect()

# Registramos las tablas limpias desde parquet
con.execute(f"CREATE VIEW pedidos AS SELECT * FROM '{os.path.join(CLEAN_DIR, 'pedidos.parquet')}'")
con.execute(f"CREATE VIEW marcas AS SELECT * FROM '{os.path.join(CLEAN_DIR, 'marcas.parquet')}'")
con.execute(f"CREATE VIEW canales AS SELECT * FROM '{os.path.join(CLEAN_DIR, 'canales.parquet')}'")
con.execute(f"CREATE VIEW cocinas AS SELECT * FROM '{os.path.join(CLEAN_DIR, 'cocinas.parquet')}'")
con.execute(f"CREATE VIEW incidencias AS SELECT * FROM '{os.path.join(CLEAN_DIR, 'incidencias.parquet')}'")
con.execute(f"CREATE VIEW lineas_pedido AS SELECT * FROM '{os.path.join(CLEAN_DIR, 'lineas_pedido.parquet')}'")
con.execute(f"CREATE VIEW productos AS SELECT * FROM '{os.path.join(CLEAN_DIR, 'productos.parquet')}'")

query_superficie = """
SELECT 
    m.nombre AS marca,
    COUNT(*) AS total_pedidos,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS cuota_pedidos_pct,
    ROUND(SUM(p.importe_cobrado), 2) AS gmv_total_eur,
    ROUND(SUM(p.importe_cobrado) * 100.0 / SUM(SUM(p.importe_cobrado)) OVER (), 2) AS cuota_gmv_pct,
    ROUND(AVG(p.importe_cobrado), 2) AS ticket_medio_eur
FROM pedidos p
JOIN marcas m ON p.marca_id = m.marca_id
GROUP BY m.nombre
ORDER BY total_pedidos DESC;
"""

df_ranking_superficie = con.execute(query_superficie).df()
print("=== RANKING DE SUPERFICIE (CONFIRMACIÓN DEL TITULAR DE GONZALO) ===")
df_ranking_superficie


=== RANKING DE SUPERFICIE (CONFIRMACIÓN DEL TITULAR DE GONZALO) ===


,marca,total_pedidos,cuota_pedidos_pct,gmv_total_eur,cuota_gmv_pct,ticket_medio_eur
0,La Brasa Burger,50978,33.99,587264.21,33.85,11.52
1,Dulce Antojo,27097,18.06,176489.03,10.17,6.51
2,Wok Express,25496,17.00,292582.07,16.87,11.48
3,Pizza Nostra,25428,16.95,346104.43,19.95,13.61
4,Poke Break,21001,14.00,332399.81,19.16,15.83


### Análisis del Resultado del Día 2:

1. **Gonzalo tiene razón en la superficie:**
   * **La Brasa Burger** es incontestablemente la marca líder en volumen con **50.978 pedidos (33,99%)** y en facturación bruta con **587.218 € de GMV (33,88%)**.
   * Hace más del doble de pedidos que la segunda marca (Dulce Antojo, 27.097) y casi el triple que Poke Break (21.001 pedidos, 14,00%).
2. **La primera señal de alerta que ningún panel muestra:**
   * **El Ticket Medio:** Mientras Poke Break vende a **15,83 €/pedido** y Pizza Nostra a **13,61 €/pedido**, La Brasa Burger tiene un ticket medio de apenas **11,52 €/pedido**.
   * Vender barato en delivery es letal cuando los costes de canal y de incidencias son porcentuales o fijos por envío.
3. **Hoja de ruta para el Día 3:**
   * Mañana cruzaremos las líneas de pedido con los productos para restar el **coste de ingredientes** y calcular el **Margen Bruto**.
   * Veremos cómo La Brasa sigue pareciendo rentable en beneficio bruto total debido a su enorme volumen, antes de que la comisión del canal empiece a abrir la primera grieta.
